## 1. Start Spark Context

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .remote("sc://spark-connect:15002") \
    .getOrCreate()

# Enables Spark's native HTML rendering for Jupyter
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 255)

spark

## 2. Show Catalog Information

In [2]:

spark.sql(""" show catalogs """)


catalog
spark_catalog


In [3]:

spark.sql(""" show schemas """)


namespace
default


In [2]:

spark.sql(""" show tables in default """)


namespace,tableName,isTemporary
default,bank_transfers,false
default,bank_users,false
default,java_edges,false
default,java_vertices,false
default,nytaxi_trips,false
default,nytaxi_trips_ext,false
default,nytaxi_zone,false


## 3. Load Data

In [5]:

df = spark.read.format("parquet").load("/examples/nytaxi/taxi_zone_lookup/")
df.createOrReplaceTempView("tmp_nytaxi_zone")

spark.sql(""" drop table if exists default.nytaxi_zone """)
spark.sql(""" CREATE TABLE IF NOT EXISTS default.nytaxi_zone USING DELTA AS SELECT * FROM tmp_nytaxi_zone """)


""


In [6]:

spark.sql(""" select * from default.nytaxi_zone """)


LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


In [7]:

df = spark.read.format("parquet").load("/examples/nytaxi/yellow_tripdata/")
df.createOrReplaceTempView("tmp_nytaxi_trips")

spark.sql(""" drop table if exists default.nytaxi_trips """)
spark.sql(""" CREATE TABLE IF NOT EXISTS default.nytaxi_trips USING DELTA AS SELECT * FROM tmp_nytaxi_trips """)


""


In [8]:

spark.sql(""" select * from default.nytaxi_trips """)


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-10-01 00:15:32,2025-10-01 01:04:03,1,17.2,2,N,132,107,1,70.0,5.0,0.5,0.0,6.94,1.0,83.44,2.5,1.75,0.75
7,2025-10-01 00:00:08,2025-10-01 00:00:08,1,5.0,1,N,107,225,1,28.2,0.0,0.5,8.49,0.0,1.0,42.44,2.5,0.0,0.75
2,2025-10-01 00:08:54,2025-10-01 00:14:44,1,2.75,1,N,263,229,1,12.8,1.0,0.5,3.71,0.0,1.0,22.26,2.5,0.0,0.75
1,2025-10-01 00:58:48,2025-10-01 01:04:40,1,1.3,1,N,211,231,2,7.9,4.25,0.5,0.0,0.0,1.0,13.65,2.5,0.0,0.75
2,2025-10-01 00:39:51,2025-10-01 00:49:40,1,2.88,1,N,230,151,1,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75
1,2025-10-01 00:30:54,2025-10-01 00:37:50,1,1.6,1,N,237,142,2,9.3,3.5,0.5,0.0,0.0,1.0,14.3,2.5,0.0,0.0
2,2025-10-01 00:10:12,2025-10-01 00:17:29,1,2.81,1,N,142,166,1,12.8,1.0,0.5,3.56,0.0,1.0,21.36,2.5,0.0,0.0
2,2025-10-01 00:48:19,2025-10-01 01:01:02,1,2.17,1,N,230,246,1,13.5,1.0,0.5,0.0,0.0,1.0,19.25,2.5,0.0,0.75
2,2025-10-01 00:08:44,2025-10-01 00:24:17,1,3.71,1,N,140,7,1,19.1,1.0,0.5,6.21,0.0,1.0,31.06,2.5,0.0,0.75
2,2025-10-01 00:25:23,2025-10-01 00:33:02,1,1.2,1,N,234,249,2,8.6,1.0,0.5,0.0,0.0,1.0,14.35,2.5,0.0,0.75


In [9]:

spark.sql("""
CREATE OR REPLACE VIEW default.nytaxi_trips_ext AS
select
	t.VendorID,
    case t.VendorID
        when 1 then 'Creative Mobile Technologies, LLC'
        when 2 then 'Curb Mobility, LLC'
        when 6 then 'Myle Technologies Inc'
        when 7 then 'Helix'
    end as VendorName,
	t.tpep_pickup_datetime,
	t.tpep_dropoff_datetime,
	t.passenger_count,
	t.trip_distance,
	t.RatecodeID,
    case t.RatecodeID
        when 1 then 'Standard rate'
        when 2 then 'JFK'
        when 3 then 'Newark'
        when 4 then 'Nassau or Westchester'
        when 5 then 'Negotiated fare'
        when 6 then 'Group ride'
        when 99 then 'Unknown'
    end as RatecodeName,
	t.store_and_fwd_flag,
	t.PULocationID,
	t.DOLocationID,
	t.payment_type,
    case t.payment_type
        when 0 then 'Flex Fare trip'
        when 1 then 'Credit card'
        when 2 then 'Cash'
        when 3 then 'No charge'
        when 4 then 'Dispute'
        when 5 then 'Unknown'
        when 6 then 'Voided trip'
    end as payment_typeName,
	t.fare_amount,
	t.extra,
	t.mta_tax,
	t.tip_amount,
	t.tolls_amount,
	t.improvement_surcharge,
	t.total_amount,
	t.congestion_surcharge,
	t.Airport_fee,
	t.cbd_congestion_fee
from
	default.nytaxi_trips t
""")


""


In [10]:

spark.sql(""" select * from default.nytaxi_trips_ext """)


VendorID,VendorName,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,RatecodeName,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,payment_typeName,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,"Creative Mobile Technologies, LLC",2025-10-01 00:15:32,2025-10-01 01:04:03,1,17.2,2,JFK,N,132,107,1,Credit card,70.0,5.0,0.5,0.0,6.94,1.0,83.44,2.5,1.75,0.75
7,Helix,2025-10-01 00:00:08,2025-10-01 00:00:08,1,5.0,1,Standard rate,N,107,225,1,Credit card,28.2,0.0,0.5,8.49,0.0,1.0,42.44,2.5,0.0,0.75
2,"Curb Mobility, LLC",2025-10-01 00:08:54,2025-10-01 00:14:44,1,2.75,1,Standard rate,N,263,229,1,Credit card,12.8,1.0,0.5,3.71,0.0,1.0,22.26,2.5,0.0,0.75
1,"Creative Mobile Technologies, LLC",2025-10-01 00:58:48,2025-10-01 01:04:40,1,1.3,1,Standard rate,N,211,231,2,Cash,7.9,4.25,0.5,0.0,0.0,1.0,13.65,2.5,0.0,0.75
2,"Curb Mobility, LLC",2025-10-01 00:39:51,2025-10-01 00:49:40,1,2.88,1,Standard rate,N,230,151,1,Credit card,14.2,1.0,0.5,3.99,0.0,1.0,23.94,2.5,0.0,0.75
1,"Creative Mobile Technologies, LLC",2025-10-01 00:30:54,2025-10-01 00:37:50,1,1.6,1,Standard rate,N,237,142,2,Cash,9.3,3.5,0.5,0.0,0.0,1.0,14.3,2.5,0.0,0.0
2,"Curb Mobility, LLC",2025-10-01 00:10:12,2025-10-01 00:17:29,1,2.81,1,Standard rate,N,142,166,1,Credit card,12.8,1.0,0.5,3.56,0.0,1.0,21.36,2.5,0.0,0.0
2,"Curb Mobility, LLC",2025-10-01 00:48:19,2025-10-01 01:01:02,1,2.17,1,Standard rate,N,230,246,1,Credit card,13.5,1.0,0.5,0.0,0.0,1.0,19.25,2.5,0.0,0.75
2,"Curb Mobility, LLC",2025-10-01 00:08:44,2025-10-01 00:24:17,1,3.71,1,Standard rate,N,140,7,1,Credit card,19.1,1.0,0.5,6.21,0.0,1.0,31.06,2.5,0.0,0.75
2,"Curb Mobility, LLC",2025-10-01 00:25:23,2025-10-01 00:33:02,1,1.2,1,Standard rate,N,234,249,2,Cash,8.6,1.0,0.5,0.0,0.0,1.0,14.35,2.5,0.0,0.75
